# Deep Learning Charity Success Predictor

This notebook walks through the complete pipeline for predicting the success of
charity funding applications using a deep neural network.

**Pipeline steps:**
1. Data loading and exploration
2. Data preprocessing (cleaning, encoding, splitting)
3. Neural network model construction
4. Model training with early stopping
5. Model evaluation and visualisation
6. Hyperparameter optimisation
7. Final model export

## 0. Setup and Imports

In [ ]:
import sys
import os

# Add the project root to the Python path so we can import our modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from data.preprocess import (
    load_data,
    clean_data,
    split_data,
    encode_features,
    preprocess_data,
    save_preprocessors,
    transform_new_data,
)
from models.model import (
    build_model,
    train_model,
    evaluate_model,
    optimize_hyperparameters,
    export_model,
    predict,
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Inline plots
%matplotlib inline

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')
print(f'Pandas version:     {pd.__version__}')

## 1. Data Loading and Exploration

In [ ]:
# Path to the charity application dataset
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'charity_data.csv')

# Load the data
df = load_data(DATA_PATH)
df.head()

In [ ]:
# Basic statistics
print(f'Dataset shape: {df.shape}')
print(f'\nColumn dtypes:\n{df.dtypes}')
print(f'\nMissing values per column:\n{df.isnull().sum()}')

In [ ]:
# Target distribution
target = 'IS_SUCCESSFUL'

fig, ax = plt.subplots(figsize=(6, 4))
df[target].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Target Distribution (IS_SUCCESSFUL)')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_xticklabels(['Unsuccessful (0)', 'Successful (1)'], rotation=0)
plt.tight_layout()
plt.show()

print(f'\nClass balance:\n{df[target].value_counts(normalize=True)}')

In [ ]:
# Explore high-cardinality categorical columns
print('APPLICATION_TYPE value counts:')
print(df['APPLICATION_TYPE'].value_counts())
print(f'\nCLASSIFICATION value counts (top 15):')
print(df['CLASSIFICATION'].value_counts().head(15))

## 2. Data Preprocessing

We use the helper functions from `data.preprocess` to:
- Drop non-beneficial ID columns (`EIN`, `NAME`)
- Bin rare categories in `APPLICATION_TYPE` (threshold < 500) and `CLASSIFICATION` (threshold < 1000)
- Split into 80/20 train / test sets (stratified)
- Scale numerical features with `StandardScaler`
- One-hot encode categorical features

In [ ]:
# Run the full preprocessing pipeline
X_train, X_test, y_train, y_test, feature_names, preprocessors = preprocess_data(
    df,
    target_column='IS_SUCCESSFUL',
    test_size=0.2,
    random_state=42,
    application_type_threshold=500,
    classification_threshold=1000,
)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test  shape: {y_test.shape}')
print(f'Number of features: {len(feature_names)}')

## 3. Build the Neural Network

Architecture:
- Input layer matching the number of features
- Hidden layers: Dense -> BatchNormalization -> Dropout
- Output layer: single neuron with sigmoid (binary classification)
- Optimizer: Adam  |  Loss: binary cross-entropy

In [ ]:
input_dim = X_train.shape[1]

model = build_model(
    input_dim=input_dim,
    hidden_layers=[128, 64],
    activation='relu',
    dropout_rate=0.2,
)

## 4. Train the Model

In [ ]:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

model, history = train_model(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    batch_size=32,
    epochs=100,
    patience=10,
    output_dir=MODELS_DIR,
)

## 5. Evaluate the Model

Generate training history curves, a confusion matrix, and a ROC curve.

In [ ]:
metrics = evaluate_model(
    model=model,
    history=history,
    X_test=X_test,
    y_test=y_test,
    output_dir=OUTPUT_DIR,
)

print('\nSummary of evaluation metrics:')
for k, v in metrics.items():
    print(f'  {k:>12s}: {v:.4f}')

## 6. Hyperparameter Optimisation (Optional)

Grid search over architectures, activations, and dropout rates.

**Note:** This can take a while. Uncomment the cell below to run.

In [ ]:
# Uncomment to run hyperparameter search

# param_grid = {
#     'hidden_layers': [[128, 64], [64, 32], [128, 64, 32]],
#     'activation': ['relu', 'tanh'],
#     'dropout_rate': [0.2, 0.3],
# }
#
# best_params = optimize_hyperparameters(
#     X_train, y_train, X_test, y_test, param_grid
# )
#
# # Retrain with best parameters
# optimised_model = build_model(
#     input_dim=input_dim,
#     hidden_layers=best_params['hidden_layers'],
#     activation=best_params['activation'],
#     dropout_rate=best_params['dropout_rate'],
# )
#
# optimised_model, opt_history = train_model(
#     optimised_model, X_train, y_train, X_test, y_test,
#     output_dir=MODELS_DIR,
# )
#
# opt_metrics = evaluate_model(
#     optimised_model, opt_history, X_test, y_test,
#     output_dir=OUTPUT_DIR,
# )

print('Hyperparameter optimisation cell (commented out by default).')

## 7. Export the Model

Save the final trained model and the preprocessing transformers so they can
be loaded later for inference on new data.

In [ ]:
# Export the model
model_path = export_model(
    model=model,
    preprocessors=preprocessors,
    output_dir=MODELS_DIR,
)

print(f'\nModel exported to: {model_path}')

## 8. Inference Example

Demonstrate how to load the exported model and run predictions on new data.

In [ ]:
from data.preprocess import load_preprocessors, transform_new_data
from models.model import load_trained_model, predict

# Load the exported artefacts
saved_model = load_trained_model(model_path)
saved_preprocessors = load_preprocessors(
    os.path.join(MODELS_DIR, 'preprocessors.pkl')
)

# Take the first 5 rows of the original data as sample input
# (drop the target column to simulate unseen data)
sample_data = df.drop(columns=['IS_SUCCESSFUL']).head(5)

# Drop ID columns that were removed during training
sample_data = sample_data.drop(
    columns=[c for c in ['EIN', 'NAME'] if c in sample_data.columns]
)

# Bin rare categories the same way training did
from data.preprocess import bin_rare_categories
sample_data = bin_rare_categories(sample_data, 'APPLICATION_TYPE', 500)
sample_data = bin_rare_categories(sample_data, 'CLASSIFICATION', 1000)

# Transform with saved preprocessors
X_sample = transform_new_data(sample_data, saved_preprocessors)

# Predict
classes, probabilities = predict(saved_model, X_sample)

print('\nInference results:')
for i, (cls, prob) in enumerate(zip(classes, probabilities)):
    label = 'Successful' if cls == 1 else 'Unsuccessful'
    print(f'  Sample {i+1}: probability={prob:.4f}  ->  {label}')

## Summary

| Step | Description |
|------|-------------|
| Data Loading | Loaded charity application CSV and inspected shape, dtypes, missing values |
| Preprocessing | Dropped IDs, binned rare categories, scaled numericals, one-hot encoded categoricals |
| Model Building | Sequential NN with Dense + BatchNorm + Dropout layers, sigmoid output |
| Training | Adam optimiser, binary cross-entropy, early stopping, checkpoint |
| Evaluation | Classification report, confusion matrix, ROC/AUC |
| Export | Saved model (.h5) and preprocessors (.pkl) for deployment |

The final model and preprocessors are saved in the `models/` directory and can
be loaded for inference on new charity application data.